# 07 — M2 Discrimination Corpus V3

**Goal:** Build a CLEAN discrimination corpus for M2 four-class router.

**V3 = MAJOR OVERHAUL.** The V1/V2 corpus had Davidson double-dipping and unfiltered HateXplain.

**Changes:**
1. `normalize_text()` on ALL text
2. **Davidson REMOVED entirely** (double-dipping with verbal harassment sources)
3. **HateXplain hatespeech FILTERED** — demographic keyword filter to ensure group-targeted only
4. **Implicit Hate CONSOLIDATED** — both threat_gap slice + disc slice → all discrimination
5. **ToxiGen ADDED** (`toxicity_human >= 2.5`, group-targeted adversarial)
6. **Stormfront hate ADDED** (`label='hate'`, white supremacist forums)
7. **HASOC HATE ADDED** (English tweets, `task_2/task2 = 'HATE'`)
8. **Waseem & Hovy DROPPED** (tweet IDs only, no actual text)
9. **TRAC-3 disc REMOVED** (Bengali/Hindi, not English)
10. OLID GRP kept (`subtask_c='GRP'`)
11. UCB MHS disc kept
12. ConvAbuse disc kept
13. Cap ~7,000 for class balance

**Design principle:** Discrimination = group-targeted hate based on demographics (race, religion, gender, nationality, etc.). NOT individual insults.

In [ ]:
# ============================================================
# CELL 1 — Setup + Shared Normalizer
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import hashlib
from collections import Counter

BASE      = Path("/content/drive/MyDrive/AEGIS_ML")
RAW       = BASE / "data" / "raw"
THREAT    = RAW / "threat_gap"
PROCESSED = BASE / "data" / "processed" / "v5"
PROCESSED.mkdir(parents=True, exist_ok=True)

def normalize_text(text: str) -> str:
    """Canonical text normalization for AEGIS pipeline."""
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\brt\s+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else None

def text_hash(t):
    return hashlib.sha256(t.strip().encode('utf-8')).hexdigest()

print("✅ Setup complete")

In [ ]:
# ============================================================
# CELL 2 — Load All Discrimination Sources
# ============================================================

frames = []
source_tracker = {}  # track rows per source

print("="*70)
print("DISCRIMINATION CORPUS V3 — LOADING SOURCES")
print("="*70)

# ══════════════════════════════════════════════════════════════
# ❌ DAVIDSON REMOVED — double-dipping (class=0 was in V1/V2
# but Davidson also feeds M1 as safe+harmful, and class=1
# was in verbal harassment). Clean separation required.
# ══════════════════════════════════════════════════════════════
print("\n❌ [REMOVED] Davidson — double-dipping, clean separation")

# ══════════════════════════════════════════════════════════════
# ❌ WASEEM & HOVY DROPPED — tweet IDs only, no actual text
# ══════════════════════════════════════════════════════════════
print("❌ [DROPPED] Waseem & Hovy — tweet IDs only, no usable text")

# ══════════════════════════════════════════════════════════════
# ❌ TRAC-3 REMOVED — Bengali/Hindi, not English
# ══════════════════════════════════════════════════════════════
print("❌ [REMOVED] TRAC-3 discrimination — not English")

# ── 1. HateXplain hatespeech (DEMOGRAPHIC KEYWORD FILTERED) ──
print("\n[1] HateXplain hatespeech (majority-vote + demographic keyword filter)")
with open(RAW / 'hatexplain.json') as f:
    hx_raw = json.load(f)

hx_rows = []
for post_id, entry in hx_raw.items():
    tokens = entry.get('post_tokens', [])
    text = ' '.join(tokens)
    # Majority vote across annotators
    annotator_labels = [a['label'] for a in entry.get('annotators', [])]
    if not annotator_labels:
        continue
    majority = Counter(annotator_labels).most_common(1)[0][0]
    hx_rows.append({'text': text, 'hx_label': majority})

df_hx = pd.DataFrame(hx_rows)
print(f"  Total parsed: {len(df_hx):,}")
print(f"  Label distribution: {df_hx['hx_label'].value_counts().to_dict()}")

# Filter for hatespeech only (NOT offensive — that goes to verbal harassment)
df_hx_hate = df_hx[df_hx['hx_label'] == 'hatespeech'].copy()
print(f"  Hatespeech rows (pre-filter): {len(df_hx_hate):,}")

# Apply normalization first
df_hx_hate['text'] = df_hx_hate['text'].apply(normalize_text)
df_hx_hate = df_hx_hate[df_hx_hate['text'].notna() & (df_hx_hate['text'].str.len() > 0)]

# Demographic keyword filter — ensure group-targeted content
DEMOGRAPHIC_KW = [
    'black', 'white', 'asian', 'muslim', 'jew', 'immigrant', 'women',
    'gay', 'trans', 'mexican', 'arab', 'refugee', 'nigger', 'fag',
    'chink', 'spic', 'kike', 'cracker', 'wetback', 'coon',
    'race', 'religion', 'country', 'your kind',
    'go back', 'all of them', 'those people',
    'islam', 'christian', 'hindu', 'jewish', 'african',
    'mexican', 'chinese', 'indian', 'latino', 'latina',
    'lesbian', 'queer', 'homo', 'dyke', 'tranny',
    'illegals', 'foreigners', 'migrants', 'invaders',
    'n word', 'racial', 'ethnic', 'minority'
]
demo_pattern = '|'.join([re.escape(kw) for kw in DEMOGRAPHIC_KW])
before_filter = len(df_hx_hate)
df_hx_hate = df_hx_hate[df_hx_hate['text'].str.contains(demo_pattern, regex=True, na=False)].copy()
print(f"  After demographic filter: {len(df_hx_hate):,} (removed {before_filter - len(df_hx_hate):,} non-demographic)")

df_hx_hate['label'] = 'discrimination'
frames.append(df_hx_hate[['text', 'label']].copy())
source_tracker['hatexplain_hate'] = len(df_hx_hate)
print(f"  ✅ {len(df_hx_hate):,} rows")

# ── 2. Implicit Hate (CONSOLIDATED — both slices) ─────────────
print("\n[2] Implicit Hate (CONSOLIDATED — threat_gap + disc slice)")
ih_frames = []

# Slice A: threat_gap/implicit_hate_threat.csv (was mislabeled as threat)
try:
    df_ih_t = pd.read_csv(THREAT / 'implicit_hate_threat.csv')
    df_ih_t['text'] = df_ih_t['text'].apply(normalize_text)
    df_ih_t = df_ih_t[df_ih_t['text'].notna() & (df_ih_t['text'].str.len() > 0)]
    ih_frames.append(df_ih_t[['text']])
    print(f"  Slice A (threat_gap, relabeled): {len(df_ih_t):,} rows")
except Exception as e:
    print(f"  ⚠️ threat_gap slice failed: {e}")

# Slice B: raw/implicit_hate_discrimination_slice.csv
try:
    df_ih_d = pd.read_csv(RAW / 'implicit_hate_discrimination_slice.csv')
    df_ih_d['text'] = df_ih_d['text'].apply(normalize_text)
    df_ih_d = df_ih_d[df_ih_d['text'].notna() & (df_ih_d['text'].str.len() > 0)]
    ih_frames.append(df_ih_d[['text']])
    print(f"  Slice B (disc slice): {len(df_ih_d):,} rows")
except Exception as e:
    print(f"  ⚠️ disc slice failed: {e}")

if ih_frames:
    df_ih = pd.concat(ih_frames, ignore_index=True)
    df_ih = df_ih.drop_duplicates(subset='text')  # dedup across slices
    df_ih['label'] = 'discrimination'
    frames.append(df_ih[['text', 'label']].copy())
    source_tracker['implicit_hate_full'] = len(df_ih)
    print(f"  ✅ {len(df_ih):,} rows (deduplicated across both slices)")

# ── 3. ToxiGen (NEW — adversarial group-targeted) ─────────────
print("\n[3] ToxiGen (toxicity_human >= 2.5, group-targeted)")
try:
    df_tg = pd.read_csv(RAW / 'toxigen_adversarial.csv')
    print(f"  Raw rows: {len(df_tg):,}")
    print(f"  Columns: {list(df_tg.columns)}")

    # Use toxicity_human if available, else toxicity_ai
    score_col = 'toxicity_human' if 'toxicity_human' in df_tg.columns else 'toxicity_ai'
    print(f"  Using score column: {score_col}")
    print(f"  Score distribution: {df_tg[score_col].describe()}")

    # Filter: high toxicity (>= 2.5 on human scale)
    df_tg[score_col] = pd.to_numeric(df_tg[score_col], errors='coerce')
    df_tg_disc = df_tg[df_tg[score_col] >= 2.5].copy()
    print(f"  After toxicity >= 2.5 filter: {len(df_tg_disc):,} rows")

    df_tg_disc['text'] = df_tg_disc['text'].apply(normalize_text)
    df_tg_disc = df_tg_disc[df_tg_disc['text'].notna() & (df_tg_disc['text'].str.len() > 0)]
    df_tg_disc['label'] = 'discrimination'
    frames.append(df_tg_disc[['text', 'label']].copy())
    source_tracker['toxigen'] = len(df_tg_disc)
    print(f"  ✅ {len(df_tg_disc):,} rows")
except Exception as e:
    print(f"  ⚠️ ToxiGen loading failed: {e}")

# ── 4. Stormfront hate (NEW) ──────────────────────────────────
print("\n[4] Stormfront (label='hate')")
try:
    df_sf = pd.read_csv(RAW / 'stormfront_hate_speech18.csv')
    df_sf_hate = df_sf[df_sf['label'] == 'hate'].copy()
    df_sf_hate['text'] = df_sf_hate['text'].apply(normalize_text)
    df_sf_hate = df_sf_hate[df_sf_hate['text'].notna() & (df_sf_hate['text'].str.len() > 0)]
    df_sf_hate['label'] = 'discrimination'
    frames.append(df_sf_hate[['text', 'label']].copy())
    source_tracker['stormfront_hate'] = len(df_sf_hate)
    print(f"  ✅ {len(df_sf_hate):,} rows")
except Exception as e:
    print(f"  ⚠️ Stormfront loading failed: {e}")

# ── 5. HASOC HATE (NEW — English tweets) ─────────────────────
print("\n[5] HASOC HATE (task_2/task2 = 'HATE')")
try:
    # Load 2019
    h19_train = pd.read_csv(RAW / 'hosoc_english/english_dataset_2019/english_dataset.tsv', sep='\t')
    h19_test  = pd.read_csv(RAW / 'hosoc_english/english_dataset_2019/hasoc2019_en_test-2919.tsv', sep='\t')
    h19 = pd.concat([h19_train, h19_test], ignore_index=True)
    h19 = h19.rename(columns={'task_2': 'task2'})
    h19_hate = h19[h19['task2'] == 'HATE'][['text']].copy()
    print(f"  HASOC 2019 HATE: {len(h19_hate):,} rows")

    # Load 2020
    h20_train = pd.read_excel(RAW / 'hosoc_english/english_dataset_2020/hasoc_2020_en_train_new.xlsx')
    h20_test  = pd.read_excel(RAW / 'hosoc_english/english_dataset_2020/hasoc_2020_en_test_new.xlsx')
    h20 = pd.concat([h20_train, h20_test], ignore_index=True)
    h20_hate = h20[h20['task2'] == 'HATE'][['text']].copy()
    print(f"  HASOC 2020 HATE: {len(h20_hate):,} rows")

    df_hasoc_hate = pd.concat([h19_hate, h20_hate], ignore_index=True)
    df_hasoc_hate['text'] = df_hasoc_hate['text'].apply(normalize_text)
    df_hasoc_hate = df_hasoc_hate[df_hasoc_hate['text'].notna() & (df_hasoc_hate['text'].str.len() > 0)]
    df_hasoc_hate['label'] = 'discrimination'
    frames.append(df_hasoc_hate[['text', 'label']].copy())
    source_tracker['hasoc_hate'] = len(df_hasoc_hate)
    print(f"  ✅ {len(df_hasoc_hate):,} rows (combined)")
except Exception as e:
    print(f"  ⚠️ HASOC HATE loading failed: {e}")

# ── 6. OLID GRP (group-targeted offensive) ────────────────────
print("\n[6] OLID GRP (subtask_c='GRP')")
try:
    # Try olid_full.tsv first
    olid_path = RAW / 'olid' / 'olid_full.tsv'
    if olid_path.exists():
        df_olid = pd.read_csv(olid_path, sep='\t')
    else:
        # Fallback: load train.csv + test.csv
        olid_train = pd.read_csv(RAW / 'olid' / 'train.csv')
        olid_test  = pd.read_csv(RAW / 'olid' / 'test.csv')
        df_olid = pd.concat([olid_train, olid_test], ignore_index=True)

    # Filter: subtask_c = 'GRP' (group-targeted)
    df_olid_grp = df_olid[df_olid['subtask_c'] == 'GRP'].copy()
    df_olid_grp['text'] = df_olid_grp['tweet'].apply(normalize_text)
    df_olid_grp = df_olid_grp[df_olid_grp['text'].notna() & (df_olid_grp['text'].str.len() > 0)]
    df_olid_grp['label'] = 'discrimination'
    frames.append(df_olid_grp[['text', 'label']].copy())
    source_tracker['olid_grp'] = len(df_olid_grp)
    print(f"  ✅ {len(df_olid_grp):,} rows")
except Exception as e:
    print(f"  ⚠️ OLID loading failed: {e}")

# ── 7. UCB MHS discrimination ─────────────────────────────────
print("\n[7] UCB MHS discrimination (pre-filtered)")
try:
    df_ucb = pd.read_csv(THREAT / 'ucb_mhs_discrimination.csv')
    df_ucb['text'] = df_ucb['text'].apply(normalize_text)
    df_ucb = df_ucb[df_ucb['text'].notna() & (df_ucb['text'].str.len() > 0)]
    df_ucb['label'] = 'discrimination'
    frames.append(df_ucb[['text', 'label']].copy())
    source_tracker['ucb_mhs_disc'] = len(df_ucb)
    print(f"  ✅ {len(df_ucb):,} rows")
except Exception as e:
    print(f"  ⚠️ UCB MHS disc loading failed: {e}")

# ── 8. ConvAbuse discrimination ───────────────────────────────
print("\n[8] ConvAbuse discrimination")
try:
    df_ca = pd.read_csv(THREAT / 'convabuse_discrimination.csv')
    df_ca['text'] = df_ca['text'].apply(normalize_text)
    df_ca = df_ca[df_ca['text'].notna() & (df_ca['text'].str.len() > 0)]
    df_ca['label'] = 'discrimination'
    frames.append(df_ca[['text', 'label']].copy())
    source_tracker['convabuse_disc'] = len(df_ca)
    print(f"  ✅ {len(df_ca):,} rows")
except:
    print("  ⚠️ File not found")

# ── COMBINE ──────────────────────────────────────────────────
print("\n" + "="*70)
print("COMBINING DISCRIMINATION SOURCES")
print("="*70)

df_disc_all = pd.concat(frames, ignore_index=True)

# Dedup
before = len(df_disc_all)
df_disc_all['text_hash'] = df_disc_all['text'].apply(text_hash)
df_disc_all = df_disc_all.drop_duplicates(subset='text_hash', keep='first').copy()
df_disc_all = df_disc_all.drop(columns='text_hash')
print(f"  Dedup: {before:,} → {len(df_disc_all):,} ({before-len(df_disc_all):,} removed)")

# Filter min words
df_disc_all = df_disc_all[df_disc_all['text'].str.split().str.len() >= 3].copy()
print(f"  After min-word filter (>= 3): {len(df_disc_all):,}")

# Source breakdown
print(f"\n📊 Source breakdown (pre-cap):")
for src, cnt in source_tracker.items():
    pct = cnt / len(df_disc_all) * 100 if len(df_disc_all) > 0 else 0
    print(f"    {src:<25} {cnt:>6,} rows ({pct:>5.1f}%)")
print(f"    {'TOTAL':<25} {len(df_disc_all):>6,} rows")

# Verify: no single source > 50%
if source_tracker:
    max_src = max(source_tracker, key=source_tracker.get)
    max_pct = source_tracker[max_src] / len(df_disc_all) * 100 if len(df_disc_all) > 0 else 0
    if max_pct > 50:
        print(f"  ⚠️ WARNING: {max_src} is {max_pct:.1f}% of corpus (> 50%)")
    else:
        print(f"  ✅ Source diversity OK (max source: {max_src} at {max_pct:.1f}%)")

# Cap at 7,000 for class balance
CAP = 7_000
if len(df_disc_all) > CAP:
    df_disc_all = df_disc_all.sample(n=CAP, random_state=42)
    print(f"\n  Capped to {CAP:,} for M2 class balance")

print(f"\n  FINAL DISCRIMINATION CORPUS: {len(df_disc_all):,} rows")

# Spot check
print("\n📌 Random sample (10 examples):")
for _, row in df_disc_all.sample(min(10, len(df_disc_all)), random_state=42).iterrows():
    print(f"  • {row['text'][:100]}")

In [ ]:
# ============================================================
# CELL 3 — Save Discrimination Corpus + Cross-Corpus Validation
# ============================================================

print("="*70)
print("SAVING DISCRIMINATION CORPUS V3")
print("="*70)

df_disc_all['language'] = 'en'
df_disc_all['source_dataset'] = 'discrimination_v3'
df_disc_all['routing_method'] = 'manual'

out_path = PROCESSED / "m2_discrimination_v3.csv"
df_disc_all.to_csv(out_path, index=False)

size_kb = out_path.stat().st_size / 1024
print(f"\n  ✅ Saved: {out_path.name} ({size_kb:.1f} KB)")
print(f"  ✅ Rows: {len(df_disc_all):,}")

# ── Normalization validation ──────────────────────────────────
print(f"\n📌 Normalization validation:")
has_upper = df_disc_all['text'].str.contains(r'[A-Z]', regex=True).sum()
has_url   = df_disc_all['text'].str.contains(r'https?://', regex=True).sum()
has_mention = df_disc_all['text'].str.contains(r'@\w+', regex=True).sum()
print(f"    Uppercase chars: {has_upper} (should be 0)")
print(f"    URLs remaining: {has_url} (should be 0)")
print(f"    @mentions remaining: {has_mention} (should be 0)")

# ── Cross-corpus overlap checks ──────────────────────────────
print(f"\n📌 Cross-corpus overlap checks:")
disc_texts = set(df_disc_all['text'])

# vs Threat
try:
    df_threat = pd.read_csv(PROCESSED / 'm2_threat_v2.csv')
    overlap_t = disc_texts & set(df_threat['text'])
    print(f"    vs Threat:             {len(overlap_t)} overlaps (should be 0)")
    if overlap_t:
        print("    ⚠️ Removing threat overlaps...")
        df_disc_all = df_disc_all[~df_disc_all['text'].isin(overlap_t)].copy()
except:
    print("    vs Threat:             (not yet built — skip)")

# vs Sexual Harassment
try:
    df_sh = pd.read_csv(PROCESSED / 'm2_sexual_harassment_v2.csv')
    overlap_sh = disc_texts & set(df_sh['text'])
    print(f"    vs Sexual Harassment:  {len(overlap_sh)} overlaps (should be 0)")
    if overlap_sh:
        print("    ⚠️ Removing sexual harassment overlaps...")
        df_disc_all = df_disc_all[~df_disc_all['text'].isin(overlap_sh)].copy()
except:
    print("    vs Sexual Harassment:  (not yet built — skip)")

# vs Verbal Harassment
try:
    df_vh = pd.read_csv(PROCESSED / 'm2_verbal_harassment_v2.csv')
    overlap_vh = disc_texts & set(df_vh['text'])
    print(f"    vs Verbal Harassment:  {len(overlap_vh)} overlaps (should be 0)")
    if overlap_vh:
        print("    ⚠️ Removing verbal harassment overlaps...")
        df_disc_all = df_disc_all[~df_disc_all['text'].isin(overlap_vh)].copy()
except:
    print("    vs Verbal Harassment:  (not yet built — skip)")

# Re-save if overlaps were removed
if any([
    'overlap_t' in dir() and overlap_t,
    'overlap_sh' in dir() and overlap_sh,
    'overlap_vh' in dir() and overlap_vh
]):
    df_disc_all.to_csv(out_path, index=False)
    print(f"\n  ✅ Re-saved after overlap removal: {len(df_disc_all):,} rows")

# ── Final summary ─────────────────────────────────────────────
print("\n" + "="*70)
print("🎉 DISCRIMINATION V3 COMPLETE")
print("="*70)
print(f"\n  📊 Total rows: {len(df_disc_all):,}")
print(f"  📊 All text normalized (lowercase, no URLs, no @mentions)")
print(f"  📊 Sources: {len(source_tracker)} unique")
for src, cnt in sorted(source_tracker.items(), key=lambda x: -x[1]):
    print(f"      {src:<25} {cnt:>6,}")
print(f"\n  📁 Saved to: {out_path}")
print(f"\n  ✅ Phase 1 corpus building is now COMPLETE!")
print(f"  📋 Next: Run all 5 notebooks in Colab, then proceed to Phase 2 (training)")